In [32]:
# Libraries and connecting R to the database
library(DBI)
library(RSQLite)
library("tidyverse")
options(repr.plot.width = 12, repr.plot.height = 8)

con <- dbConnect(RSQLite::SQLite(), "C:/Users/atd29/OneDrive/Documents/University/ECONOMIC MASTERS/ECON1205/econ1205_CW_database.db")

In [4]:
# List all tables in the db
dbListTables(con)

[1] "carbon_gdp_intensity"     "consumer_price_inflation"
[3] "countries"                "gdp"                     
[5] "gdp_per_capita"           "gdp_per_capita_growth"   
[7] "net_energy_imports"

In [5]:
# View column names of a specific table
dbListFields(con, "gdp")

[1] "Country Code"  "Series Name"   "Series Code"   "2022 [YR2022]"

In [34]:
# A list of all table names in the database
table_names <- dbListTables(con)

# Fixing the errors in the 2022 [YR2022] column ~ converting them into numerical
all_tables <- table_names |>
  set_names() |>
  map(\(tbl_name) {
      suppressWarnings(
          dbReadTable(con, tbl_name, check.names = FALSE) |>
          mutate(across(contains("[YR"), as.numeric))
      )
  })

all_tables$gdp # For accessing individual tables in the database

dbDisconnect(con)

Country Code,Series Name,Series Code,2022 [YR2022]
<chr>,<chr>,<chr>,<dbl>
AFG,GDP (current US$),NY.GDP.MKTP.CD,1.449724e+10
ALB,GDP (current US$),NY.GDP.MKTP.CD,1.901725e+10
DZA,GDP (current US$),NY.GDP.MKTP.CD,2.255816e+11
ASM,GDP (current US$),NY.GDP.MKTP.CD,8.710000e+08
AND,GDP (current US$),NY.GDP.MKTP.CD,3.380613e+09
AGO,GDP (current US$),NY.GDP.MKTP.CD,1.282340e+11
ATG,GDP (current US$),NY.GDP.MKTP.CD,1.857115e+09
ARG,GDP (current US$),NY.GDP.MKTP.CD,6.339938e+11
ARM,GDP (current US$),NY.GDP.MKTP.CD,1.951351e+10


### Regression 1 ~ gdp and carbon intensity

In [33]:
# Renaming the data in the tables I want to use & filtering out rows that do not have a 
# 3 letter code in the country code column.
gdp_df <- all_tables$gdp |> 
    filter(str_detect(`Country Code`, "^[A-Z]{3}$")) |>
    select(`Country Code`, gdp = `2022 [YR2022]`)

carbon_df <- all_tables$carbon_gdp_intensity |>
    filter(str_detect(`Country Code`, "^[A-Z]{3}$")) |>
    select(`Country Code`, carbon_intensity = `2022 [YR2022]`)

# Joining both tables together (like in python)
combined_data1 <- inner_join(gdp_df, carbon_df, by = "Country Code")
summary(combined_data1)

# Running the linear regression (regressing carbon_intensity on gdp)
set.seed(42)
model1 <- lm(carbon_intensity ~ gdp, data = combined_data1)
summary(model1)

    Country Code      gdp            carbon_intensity
 Length   :266   Min.   :0.000e+00   Min.   :0.0000  
 N.unique :266   1st Qu.:9.578e+09   1st Qu.:0.1632  
 N.blank  :  0   Median :6.108e+10   Median :0.3158  
 Min.nchar:  3   Mean   :3.201e+12   Mean   :0.4276  
 Max.nchar:  3   3rd Qu.:6.109e+11   3rd Qu.:0.5742  
                 Max.   :1.029e+14   Max.   :6.1077  


Call:
lm(formula = carbon_intensity ~ gdp, data = combined_data1)

Residuals:
    Min      1Q  Median      3Q     Max 
-0.4249 -0.2665 -0.1090  0.1469  5.6829 

Coefficients:
             Estimate Std. Error t value Pr(>|t|)    
(Intercept) 4.248e-01  3.246e-02  13.087   <2e-16 ***
gdp         8.719e-16  2.829e-15   0.308    0.758    
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.5084 on 264 degrees of freedom
Multiple R-squared:  0.0003597,	Adjusted R-squared:  -0.003427 
F-statistic: 0.095 on 1 and 264 DF,  p-value: 0.7582


### Regression 2 ~ gdp per capita and gdp per capita growth

In [36]:
gdp_pc_df <- all_tables$gdp_per_capita |> 
    filter(str_detect(`Country Code`, "^[A-Z]{3}$")) |>
    select(`Country Code`, gdp_pc = `2022 [YR2022]`)

gdp_pcg_df <- all_tables$gdp_per_capita_growth |>
    filter(str_detect(`Country Code`, "^[A-Z]{3}$")) |>
    select(`Country Code`, gdp_pcg = `2022 [YR2022]`)

# Joining both tables together
combined_data2 <- inner_join(gdp_pc_df, gdp_pcg_df, by = "Country Code")
summary(combined_data2)

# Running the linear regression 
set.seed(42)
model2 <- lm(gdp_pc ~ gdp_pcg, data = combined_data2)
summary(model2)

    Country Code     gdp_pc          gdp_pcg       
 Length   :266   Min.   :     0   Min.   :-22.746  
 N.unique :266   1st Qu.:  2325   1st Qu.:  1.274  
 N.blank  :  0   Median :  6989   Median :  2.871  
 Min.nchar:  3   Mean   : 18869   Mean   :  3.180  
 Max.nchar:  3   3rd Qu.: 22747   3rd Qu.:  4.783  
                 Max.   :226052   Max.   : 62.111  


Call:
lm(formula = gdp_pc ~ gdp_pcg, data = combined_data2)

Residuals:
   Min     1Q Median     3Q    Max 
-19341 -16465 -11692   2767 205802 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept)  18235.7     2003.3   9.103   <2e-16 ***
gdp_pcg        199.2      295.9   0.673    0.501    
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 28840 on 264 degrees of freedom
Multiple R-squared:  0.001713,	Adjusted R-squared:  -0.002068 
F-statistic: 0.4531 on 1 and 264 DF,  p-value: 0.5014


### Regression 3 ~ net energy imports and gdp per capita growth

In [37]:
energy_df <- all_tables$net_energy_imports |> 
    filter(str_detect(`Country Code`, "^[A-Z]{3}$")) |>
    select(`Country Code`, net_energy_imports = `2022 [YR2022]`)

# gdp_pcg_df <- all_tables$gdp_per_capita_growth |>
    # filter(str_detect(`Country Code`, "^[A-Z]{3}$")) |>
    # select(`Country Code`, gdp_pcg = `2022 [YR2022]`)

# Joining both tables together
combined_data3 <- inner_join(energy_df, gdp_pcg_df, by = "Country Code")
summary(combined_data3)

# Running the linear regression 
set.seed(42)
model3 <- lm(net_energy_imports ~ gdp_pcg, data = combined_data3)
summary(model3)

    Country Code net_energy_imports    gdp_pcg       
 Length   :266   Min.   :-731.00    Min.   :-22.746  
 N.unique :266   1st Qu.: -28.13    1st Qu.:  1.274  
 N.blank  :  0   Median :  14.93    Median :  2.871  
 Min.nchar:  3   Mean   : -10.69    Mean   :  3.180  
 Max.nchar:  3   3rd Qu.:  52.91    3rd Qu.:  4.783  
                 Max.   : 389.33    Max.   : 62.111  
                 NAs    :81                          


Call:
lm(formula = net_energy_imports ~ gdp_pcg, data = combined_data3)

Residuals:
    Min      1Q  Median      3Q     Max 
-721.43  -12.59   25.64   62.91  409.87 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)  
(Intercept)  -20.865     12.038  -1.733   0.0847 .
gdp_pcg        3.589      2.660   1.349   0.1789  
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 127.6 on 183 degrees of freedom
  (81 observations deleted due to missingness)
Multiple R-squared:  0.009849,	Adjusted R-squared:  0.004439 
F-statistic:  1.82 on 1 and 183 DF,  p-value: 0.1789


### Regression 4 ~ net energy imports and consumer price inflation

In [40]:
# energy_df <- all_tables$net_energy_imports |> 
    # filter(str_detect(`Country Code`, "^[A-Z]{3}$")) |>
    # select(`Country Code`, net_energy_imports = `2022 [YR2022]`)

inflation_df <- all_tables$consumer_price_inflation |>
    filter(str_detect(`Country Code`, "^[A-Z]{3}$")) |>
    select(`Country Code`, inflation = `2022 [YR2022]`)

# Joining both tables together
combined_data4 <- inner_join(energy_df, inflation_df, by = "Country Code")
summary(combined_data4)

# Running the linear regression 
set.seed(42)
model4 <- lm(net_energy_imports ~ inflation, data = combined_data4)
summary(model4)

    Country Code net_energy_imports   inflation      
 Length   :266   Min.   :-731.00    Min.   : -6.687  
 N.unique :266   1st Qu.: -28.13    1st Qu.:  4.261  
 N.blank  :  0   Median :  14.93    Median :  7.626  
 Min.nchar:  3   Mean   : -10.69    Mean   : 10.070  
 Max.nchar:  3   3rd Qu.:  52.91    3rd Qu.: 10.279  
                 Max.   : 389.33    Max.   :171.205  
                 NAs    :81                          


Call:
lm(formula = net_energy_imports ~ inflation, data = combined_data4)

Residuals:
    Min      1Q  Median      3Q     Max 
-715.52  -26.31   26.50   66.47  404.51 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)  
(Intercept) -19.9080    11.0368  -1.804   0.0729 .
inflation     0.7674     0.4861   1.579   0.1161  
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 127.4 on 183 degrees of freedom
  (81 observations deleted due to missingness)
Multiple R-squared:  0.01344,	Adjusted R-squared:  0.008048 
F-statistic: 2.493 on 1 and 183 DF,  p-value: 0.1161


# Interpretation of results
Regression 1, GDP and Carbon Intensity of GDP:
- The Intercept figure: (4.248e-01) means that if a country had a GDP of 0 USD the carbon intensity would be "0.4248".
- The "gdp" estimate: (8.719e-16) means that for an increase in 1 USD of gdp, the carbon intensity would increase by virtually 0.
- Multiple R-squared: (0.0003597) means that gdp accounts for 0.036% of variation in carbon intensity.
- Residual Values: The target for any prediction would be for a residual value of 0, as that would be an exact prediction. The maximum underestimation was 5.68 away from the actual value. 
- Conclusion: Virtually no relationship between GDP and Carbon Intensity found by this regression. 

Regression 2, GDP per Capita and GDP per Capita Growth:
- The Intercept figure: (18,235.7) means that if a country had a growth of of 0% the GDP per Capita would be 18,235.70 USD. 
- The "gdp_pc" estimate: (199.2) means that for a growth increase of 1%, GDP per Capita would increase by 199.20 USD
- Multiple R-squared: (0.001713) means that the growth rate accounts for 0.1713% of variance in GDP per Capita across countries, virtually 0. 
- Residual Values: The maximum underestimation was 205,802 USD away from the actual value. On average, the estimation is off by 28,840 USD (the Residual Standard Error), which is huge considering the median income is only 6,989 USD. 
- Conclusion: This regression was terrible. No relationship found. 

Regression 3, Net Energy Imports and GDP per Capita Growth:
- The Intercept figure: (-20.865) means that if a country had 0% economic growth it would have net energy imports of -20.865% ~ a net exporter. 
- The "gdp_pcg" estimate: (3.589) means that for a 1% increase in GDP per Capita Growth, net energy imports are estimated to increase by 3.59%.
- Multiple R-squared: (0.009849) means that economic growth explains 0.9849% of variations in net energy imports ~ virtually insignificant, like the results from the previous two regressions.
- Residual Values: The maximum underestimation was 409.87% away from the actual value. The average error was 127.6%.
- Conclusion: This regression was slightly better than its predecessors but still did not reveal a significant relationship between the variables. Weak relationship (insignificant). 

Regression 4, Net Energy Imports and Consumer Price Inflation:
- The Intercept figure: (-19.908) means that if a country had 0% consumer inflation they would have estimated net energy imports of -19.908% ~ making them a net energy exporter. 
- The "inflation" estimate: (0.7674) means that for a 1% increase in inflation, net energy imports increase by 0.7674%. 
- Multiple R-squared: (0.01344) means that inflation explains 1.34% of the variation in net energy imports across countries. 
- Residual Values: The maximum underestimation was 404.51% away from the actual value. The average error was 127.4%.
- Conclusion: Weak relationship (insignificant). 

Conclusions
The results across the board point towards non-existant ond insignificant relationship between the variables I have selected to conduct my regressions on. 